# Residential location: which dwelling a moving household chooses within the PUMA

## What this model answers

Within the chosen PUMA, which dwelling does a moving household take? The model compares the chosen dwelling with 29 other dwellings of the same PUMA by cost, size, age and structure type, with effects that change with income. It is the lower level of the two-level residential model.

**Why 29 other dwellings.** A PUMA has thousands of dwellings, and the ACS sample holds only some of them, so the model compares the chosen dwelling with 29 others drawn uniformly at random from the same pool: the PUMA's dwellings of households that moved in within 12 months, the pool the chosen one comes from. Uniform random draws give unbiased estimates (McFadden 1978); only precision is lost. Re-estimating with a second draw of 30, and with 60 and 100 alternatives, changes no coefficient by more than 0.67 robust standard errors, so 30 are enough. In SILO a moving household chooses among 30 vacant dwellings drawn at random in the chosen PUMA (all of them if fewer).

| | |
|---|---|
| Choice | The dwelling moved into |
| Alternatives | The chosen dwelling and 29 dwellings drawn at random from the same PUMA's dwellings of recent movers |
| Decision makers | Households that moved into their dwelling in the last 12 months |
| Data | ACS PUMS 2014-2018 (5-year), DC, DE, MD, PA, VA, WV |
| Estimation sample | 15,163 movers |
| Hold-out sample | 3,820 movers |

## How to run

- Kernel: *Python (biogeme)* (`~/.venvs/biogeme`, Biogeme 3.3); the data rebuild also needs `python3` with geopandas. Then *Run All*.
- `RUN_DATA_PREP`: rebuild the estimation tables in a scratch folder and compare them with the shipped ones.
- `RUN_CHECK`: log-likelihood at the article coefficients, no estimation (about 0.1 min).
- `RUN_ESTIMATION`: run the Biogeme estimation (off by default; the results in `output/` are read instead).
- `RUN_HOLDOUT`: log-likelihood of the hold-out records.
- Figures of the hold-out check are read from `output/holdout_*` (copied there by the notebook builder).

## Contents

Article table: `est_resloc_dwelling`. This notebook goes from the raw records to the estimates of this model:

1. Raw data
2. Data preparation: filters from the raw records to the estimation sample (records kept and dropped), rebuild check
3. Estimation data and choice set
4. Descriptive statistics and figures; 4.1 every explanatory variable of the model in the data; 4.2 new dwellings, bedrooms and housing cost
5. Model specification (the folder's `model/estimate_residential_dwelling.py`, unchanged)
6. Correlation of the explanatory variables (heatmaps, VIF)
7. Log-likelihood at the article coefficients
8. Biogeme estimation
9. Results: article format, fit, comparison with the article, correlation of the estimates
10. Hold-out sample

11. Glossary

## Model at a glance

**1. Who decides.** Households that moved into their dwelling in the last 12 months. Data: ACS PUMS 2014-2018 (5-year), DC, DE, MD, PA, VA, WV. Estimation sample: 15,163 movers; hold-out: 3,820.

**2. What is chosen (dependent variable).** The dwelling moved into. Alternatives: The chosen dwelling and 29 dwellings drawn at random from the same PUMA's dwellings of recent movers.

**3. Explanatory variables** (signs of the article estimates, `model/reference_estimates.csv`):

| Group | Variable | What it measures | Sign |
|---|---|---|---|
| Affordability | Cost above 30 % of income | max(0, 12 x monthly cost / income - 0.30) | − |
| Affordability | Cost above 30 % x ln income | shift per unit ln(income / $50k) | − (shift per unit ln income) |
| Quality | ln monthly housing cost | price signal of dwelling quality | + |
| Quality | ln cost x ln income | shift per unit ln(income / $50k) | + (shift per unit ln income) |
| Size and age | Bedroom shortfall | bedrooms below household size (at most 5) | − |
| Size and age | Bedroom shortfall x ln income | shift per unit ln(income / $50k) | − (shift per unit ln income) |
| Size and age | Bedroom excess | bedrooms above household size | − |
| Size and age | Bedroom excess x ln income | shift per unit ln(income / $50k) | + (shift per unit ln income) |
| Size and age | Built in the last five years | new dwelling | − |
| Size and age | New dwelling x ln income | shift per unit ln(income / $50k) | + (shift per unit ln income) |
| Structure type (vs single-family detached) | Single-family attached | constant | + |
| Structure type (vs single-family detached) | Single-family attached x ln income | shift per unit ln(income / $50k) | − (shift per unit ln income) |
| Structure type (vs single-family detached) | Multifamily 2-4 units | constant | − |
| Structure type (vs single-family detached) | Multifamily 2-4 x ln income | shift per unit ln(income / $50k) | − (shift per unit ln income) |
| Structure type (vs single-family detached) | Multifamily 5+ units | constant | − |
| Structure type (vs single-family detached) | Multifamily 5+ x ln income | shift per unit ln(income / $50k) | − (shift per unit ln income) |
| Structure type (vs single-family detached) | Mobile home or other | constant | − |
| Structure type (vs single-family detached) | Mobile home x ln income | shift per unit ln(income / $50k) | − (shift per unit ln income) |

**4. How household traits enter.** Household traits (income, size, age, cars, children) are the same for every alternative, so on their own they cannot change which alternative is chosen. They enter only as interactions with attributes of the alternatives (for example density x single-person household, car time x income).

**5. How the two levels connect.** The PUMA level contains the inclusive value of the dwelling level (ln of the mean of exp(V) over 200 sampled dwellings of the PUMA) with coefficient 1, and ln M as a separate term, M = dwelling stock x fixed turnover rate of the PUMA. With the coefficient at 1 the two levels are one logit over all dwellings of the area.

**6. How to read a coefficient.** A positive coefficient makes the alternative more likely, a negative one less likely. exp(coefficient) is the factor by which the odds change for one unit more of the variable.
 Income enters as ln(income / $50k), so a main coefficient is that of a $50k household and the x ln income term shifts it: Multifamily 2-4 units is +0.11 at $25k, −0.13 at $50k and −0.36 at $100k.

In [ ]:
# switches (see How to run) and set-up: imports, folders, helper modules
RUN_DATA_PREP  = True    # rebuild the estimation sample and tables into output/rebuild_check and compare with the shipped tables (section 2)
RUN_CHECK      = True    # log-likelihood at the article coefficients
RUN_ESTIMATION = False   # Biogeme estimation, written to output/ (False: results are read from output/ if there are any)
RUN_HOLDOUT    = True    # log-likelihood of the hold-out sample (at your estimates if there are any, else at the article coefficients)

import os, sys, json, glob, inspect, importlib, time, re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import biogeme.biogeme as bio
from biogeme.parameters import Parameters
from biogeme.results_processing import EstimationResults

FOLDER = os.path.abspath("")                       # this model folder (VS Code starts the notebook here)
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt; import nb_reader as rd; import figs as fg
fg = importlib.reload(fg)
from IPython.display import Markdown
sys.path.insert(0, os.path.join(FOLDER, "model"))
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
OUT = os.path.join(FOLDER, "output"); os.makedirs(OUT, exist_ok=True)
NAME = "residential_dwelling"; MODEL_FOLDER = "02_residential_dwelling"
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("model folder:", os.path.basename(FOLDER))

### Sampled-alternatives check

Why the sampled alternatives are enough: the model was re-estimated with a second random draw of 30, and with 60 and 100 alternatives (each draw also redraws the 30 alternatives (chosen + 29 sampled)). Read from `02_residential_dwelling/output/sample_size_coef.csv` and `sample_size_fit.csv` (the check runs in the project, not in this package).

In [ ]:
# change of every coefficient against the 30-alternative estimate, in its robust s.e. (grey band: within 1 s.e.), and the fits
SS = os.path.join(FOLDER, "..", "02_residential_dwelling", "output")
fg.sample_size_figure(os.path.join(SS, "sample_size_coef.csv"), 'dwelling'); plt.show()
ssf = pd.read_csv(os.path.join(SS, "sample_size_fit.csv")); n_est = int(RUN["check"]["N"])
ssf["N"] = n_est; ssf["LL per mover (dwelling level)"] = ssf["dwelling_LL"] / n_est
display(ssf[["case", "J", "N", "LL per mover (dwelling level)"]].round(4))

How to read it: all points inside the grey band (within one standard error) mean the results do not depend on the number of sampled dwellings. The dwelling-level log-likelihood falls with more alternatives because the choice set is larger; it is not a comparison of fit.

## 1. Raw data

*Where the records come from. Look for: the files and the record counts.*

In [ ]:
# Raw data
SH = os.path.abspath(os.path.join(FOLDER, "..", "00_shared_data")); CODE = os.path.join(FOLDER, "input", "code")
RAW = os.path.join(FOLDER, "input", "data", "raw"); REBUILD = os.path.join(OUT, "rebuild_check")
ZS = pd.read_csv(os.path.join(SH, "zone_system", "zoneSystem.csv"))[["ZoneId", "COUNTYFIPS", "JUR_NAME", "PUMA_10"]].rename(columns={"ZoneId": "zone"})
ZS["area_group"] = nt.area_group_of_county(ZS.COUNTYFIPS)

def rebuild_tables(python=sys.executable, pre=""):
    """build_estimation_data.main() of input/code with its output folder set to a scratch folder; returns the comparison with the
    shipped estimation and hold-out tables."""
    out = os.path.join(REBUILD, "data"); [os.makedirs(os.path.join(out, h), exist_ok=True) for h in ("estimation", "holdout")]
    nt.run(f'{python} -c "import build_estimation_data as B; {pre}B.OUT = {out!r}; B.main()"', cwd=CODE)
    return pd.concat([nt.compare_trees(os.path.join(out, h), os.path.join(FOLDER, "input", "data", h)).assign(half=h) for h in ("estimation", "holdout")],
                     ignore_index=True)

### 1.1 ACS PUMS 2014-2018 (5-year), six states

The residential models are estimated on households of the ACS Public Use Microdata Sample (household and person files of
DC, DE, MD, PA, VA, WV; `../00_shared_data/acs_pums_2014_2018_5yr`) that moved into their dwelling within the last
12 months. Origin (migration PUMA of the householder) and destination (PUMA of the dwelling) are PUMAs, mapped to the
SILO counties and zones by the geography tables of `../00_shared_data/resloc_work` (zones without a 2010 tract internal point
by their centroid: `geo_puma_zone_all.csv`, `geo_mig_zone_all.csv`). The householder's race (PUMS `RAC1P`, `HISP`) enters
only the own-race share of the PUMA level.

In [ ]:
# 1.1 ACS PUMS 2014-2018 (5-year), six states
display(nt.files_table(os.path.join(SH, "acs_pums_2014_2018_5yr")))
MOV = pd.read_parquet(os.path.join(SH, "resloc_work", "pums_movers_est.parquet"))
POOL = pd.read_parquet(os.path.join(SH, "resloc_work", "pums_pool_est.parquet"))
print(f"mover households: {len(MOV):,} (estimation {int((MOV.half == 'estimation').sum()):,}, hold-out {int((MOV.half == 'validation').sum()):,}); "
      f"dwellings occupied by recent movers (pool of alternatives): {len(POOL):,}")
nt.no_race(MOV).head()

## 2. Data preparation

*From the raw records to the estimation sample. Look for: the records kept and dropped at each filter. With RUN_DATA_PREP the rebuilt tables must equal the shipped ones.*

### 2.1 From PUMS records to mover households

`input/code/s03_pums.py` reads the 12 PUMS files and keeps the mover households below. The split into an estimation
(80 %) and a hold-out half (20 %) is fixed by a hash of the household serial number (`md5(SERIALNO) mod 5 = 0` = hold-out).
The reading and the filters are run here with the script's own functions.

In [ ]:
# ACS PUMS 2014-2018 of DC, DE, MD, PA, VA, WV: the reading and filters of s03_pums.main('est'), mirrored
sys.path.insert(0, CODE)
import s03_pums as S3
P, H = S3.load("est")
f = nt.Funnel("PUMS household records (6 states)", len(H))
H = H[(H.TYPE == 1) & (H.NP > 0)].copy(); f.add("households (not group quarters, not vacant)", len(H), "group quarters and vacant units")
rel = P.RELP; hd = P[rel == 0].drop_duplicates("SERIALNO").set_index("SERIALNO")
H = H.set_index("SERIALNO"); H["mig"] = hd.MIG.reindex(H.index)
H["mig7"] = (hd.MIGSP.reindex(H.index).fillna(0) * 100000 + hd.MIGPUMA.reindex(H.index).fillna(0)).astype(int)
H["puma7"] = H.ST * 100000 + H.PUMA; H["dtype"] = S3.bld_type(H.BLD)
cov = pd.read_csv(os.path.join(SH, "resloc_work", "geo_puma_region.csv")).groupby("puma7").coverage.first()
mcov = pd.read_csv(os.path.join(SH, "resloc_work", "geo_mig_zone.csv")).groupby("mig7").coverage.first()
H["dest_cov"] = H.puma7.map(cov).fillna(0); H["orig_cov"] = H.mig7.map(mcov).fillna(0)
x = H[H.MV == 1]; f.add("moved in within the last 12 months", len(x), "not a recent mover")
x = x[x.mig == 3]; f.add("householder lived in another U.S. house a year ago", len(x), "same house, or abroad")
x = x[x.dest_cov > 0]; f.add("destination PUMA in the zone system", len(x), "PUMA outside the study area")
x = x[x.orig_cov > 0]; f.add("origin MIGPUMA in the zone system", len(x), "moved in from outside the study area")
x = x[x.dtype != "NA"]; f.add("known structure type (mover households)", len(x), "boat, RV, van")
MOV = pd.read_parquet(os.path.join(SH, "resloc_work", "pums_movers_est.parquet"))
f.add("estimation half (80 %, hash of SERIALNO)", int((MOV.half == "estimation").sum()), "hold-out half (20 %)")
display(f.table()); nt.funnel_plot(f, "households")
print("mirrored mover households equal the shipped pums_movers_est.parquet:", sorted(x.index) == sorted(MOV.SERIALNO))

### 2.2 Rebuild check

`s03_pums.main('est')` is run with its output folder set to a scratch folder (movers and pool files compared with the shipped ones); then `build_estimation_data.py` writes the estimation tables into a scratch folder and they are compared with the shipped ones (the script also asserts the log-likelihood of the original run).

In [ ]:
# 2.2 Rebuild check
if RUN_DATA_PREP:
    import shutil
    W3 = os.path.join(REBUILD, "resloc_work"); os.makedirs(W3, exist_ok=True)
    for f_ in ["geo_puma_region.csv", "geo_mig_zone.csv"]:
        shutil.copy2(os.path.join(SH, "resloc_work", f_), W3)
    nt.run(f'{sys.executable} -c "import s03_pums as S; S.WORK = {W3!r}; S.main(\\"est\\")"', cwd=CODE, quiet=True)
    res_p = pd.concat([nt.compare_frames(pd.read_parquet(os.path.join(W3, f_)), pd.read_parquet(os.path.join(SH, "resloc_work", f_)), name=f_)
                       for f_ in ["pums_movers_est.parquet", "pums_pool_est.parquet"]], ignore_index=True)
    display(res_p)
else:
    print("skipped (RUN_DATA_PREP = False)")

if RUN_DATA_PREP:
    t0 = time.time()
    res = rebuild_tables(sys.executable)
    print(f"build: {(time.time() - t0) / 60:.1f} min"); display(res)
    nt.rebuild_verdict(pd.concat([res_p, res], ignore_index=True), glob.glob(os.path.join(FOLDER, "model", "estimate_*.py"))[0])

### Data sources

Where each source comes from and where it goes (paths from `manual_estimation/`). *verify*: no download link is recorded in the project. The full download steps are in `INSTRUCTIONS.pdf`.

**ACS PUMS 2014-2018 5-year, household and person files** — U.S. Census Bureau
- Where: https://www2.census.gov/programs-surveys/acs/data/pums/2018/5-Year/
- How: Download csv_h<st>.zip and csv_p<st>.zip for st = dc, de, md, pa, va, wv from the 2018 5-Year folder (CSV, not SAS). Variables used: SERIALNO, ST, PUMA, MIGPUMA, MIGSP, POWPUMA, POWSP, MIG, MV, RELP, AGEP, ESR, JWMNP, SCHG, NP, TYPE, BDSP, BLD, TEN, GRNTP, SMOCP, VALP, HINCP, VEH, YBL, WGTP; ADJINC (income adjustment) and ADJHSG (housing-cost adjustment) are columns of the same files.
- Put in: 00_shared_data/raw/acs_pums_2014_2018_5yr/
- Files: `csv_hdc.zip`, `csv_hde.zip`, `csv_hmd.zip`, `csv_hpa.zip`, `csv_hva.zip`, `csv_hwv.zip`, `csv_pdc.zip`, `csv_pde.zip`, `csv_pmd.zip`, `csv_ppa.zip`, `csv_pva.zip`, `csv_pwv.zip`

**2010 Census tract to 2010 PUMA relationship file** — U.S. Census Bureau
- Where: https://www2.census.gov/geo/docs/maps-data/data/rel/2010_Census_Tract_to_2010_PUMA.txt
- How: Download the national text file.
- Put in: 00_shared_data/raw/geo/
- Files: `2010_Census_Tract_to_2010_PUMA.txt`

**PUMA to MIGPUMA and POWPUMA crosswalk (2010 geography)** — IPUMS USA
- Where: https://usa.ipums.org/usa/resources/volii/puma_migpuma1_pwpuma00.xls
- How: Download the xls.
- Put in: 00_shared_data/raw/geo/
- Files: `puma_migpuma1_pwpuma00.xls`

**Census Gazetteer, national tract file (land area, internal points)** — U.S. Census Bureau
- Where: https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html
- How: Gazetteer files, census tracts, national file (the project uses Gaz_tracts_national.txt).
- Put in: 00_shared_data/raw/geo/
- Files: `Gaz_tracts_national.txt`

**MSTM zone system, zone polygons and peak car skim (Maryland Statewide Transportation Model)** — Maryland Department of Transportation State Highway Administration (MDOT SHA), provided to the project
- Where: https://roads.maryland.gov
- How: Not a public download: the 1,588-zone MSTM system (zoneSystem.csv), zone polygons (SMZ_RMZ_02152011inMSTM_EPSG26918.shp) and the AM-peak auto travel-time skim were provided by MDOT. Request them from MDOT SHA (MSTM / travel forecasting).
- Put in: 00_shared_data/raw/mstm/
- Files: `zoneSystem.csv`, `SMZ_RMZ_02152011inMSTM_EPSG26918.shp/.shx/.dbf/.prj`, `auto_peak_time.csv.gz`

**CPI-U, U.S. city average, all items (annual averages 2014-2023)** — U.S. Bureau of Labor Statistics
- Where: https://data.bls.gov/timeseries/CUUR0000SA0
- How: Series CUUR0000SA0, annual averages; the values are typed in rc.py (CPI dict) and used to express incomes in 2016 dollars.
- Put in: 00_shared_data/raw/cpi/

## 3. Estimation data and choice set

*The table the model reads. Look for: the columns used and what they mean (data dictionary).*

Alternatives: the chosen dwelling (alternative 0) and 29 dwellings drawn without replacement from the dwellings of the same PUMA occupied by households that moved in within 12 months (`cost_<j>`, `bedrooms_<j>`, `type_<j>`).

The table exactly as used in the estimation, and its data dictionary.

In [ ]:
# Estimation data and choice set
est = os.path.join(FOLDER, "input", "data", "estimation")
display(nt.files_table(est))
dd = pd.read_csv(os.path.join(est, "data_dictionary.csv"))
dd

In [ ]:
# Estimation data and choice set
import estimate_residential_dwelling as M
df = pd.read_parquet(M.DATA)
print(f"{df.shape[0]:,} rows (decision makers), {df.shape[1]:,} columns")
df.head()

## 4. Descriptive statistics

*What the sample looks like. Look for: plausible shares and maps, and the chosen alternatives against the available ones.*

In [ ]:
# Chosen dwellings: structure type, monthly housing cost and bedrooms; the 29 sampled alternatives come from the
TY = ["SFD", "SFA", "MF234", "MF5plus", "MH"]
HO = pd.read_parquet(os.path.join(FOLDER, "input", "data", "holdout", "dwelling_holdout.parquet"), columns=["type_0", "cost_0", "bedrooms_0"])
fig, ax = nt.panels(3, ncol=3, w=4.3, h=2.9)
c = df.type_0.value_counts().reindex(TY); nt.bar(100 * c / c.sum(), "structure type of the chosen dwelling", "share (%)", ax=ax[0])
nt.hist(df.cost_0.clip(upper=4000), "monthly housing cost of the chosen dwelling (2016 $)", "households", bins=50, ax=ax[1])
c = df.bedrooms_0.clip(upper=5).value_counts().sort_index(); nt.bar(100 * c / c.sum(), "bedrooms (5 = 5+)", "share (%)", ax=ax[2])
plt.tight_layout(); plt.show()
alt = pd.DataFrame({"chosen dwelling": [df.cost_0.median(), df.bedrooms_0.mean()],
                    "sampled alternatives": [np.nanmedian(df[[f"cost_{j}" for j in range(1, 30)]].to_numpy()), np.nanmean(df[[f"bedrooms_{j}" for j in range(1, 30)]].to_numpy())]},
                   index=["median monthly cost ($)", "mean bedrooms"]).round(1)
alt

Chosen dwellings: structure type, monthly housing cost and bedrooms; the 29 sampled alternatives come from the same PUMA's pool of recently occupied dwellings.

In [ ]:
# chosen against sampled dwellings: cost share of income and bedrooms minus household size
cs = lambda j: 12 * df[f"cost_{j}"] / np.maximum(df.income_2016usd, 5000)
alts_cs = np.concatenate([cs(j).to_numpy() for j in range(1, 30)]); alts_bd = np.concatenate([(df[f"bedrooms_{j}"] - df.persons.clip(upper=5)).to_numpy() for j in range(1, 30)])
nt.chosen_vs_available([("Housing cost / income", cs(0).clip(upper=1.5), np.clip(alts_cs, None, 1.5)),
                        ("Bedrooms minus household size", df.bedrooms_0 - df.persons.clip(upper=5), alts_bd)], "Chosen against the 29 sampled dwellings, estimation movers")

How to read it: line = chosen dwellings, grey = the 29 sampled dwellings; the further apart the two, the more the variable separates the choices.

In [ ]:
# Mover households (persons, income, children, age), their PUMAs, and the shares of the two halves (estimation 8
fig, ax = nt.panels(4, ncol=4, w=3.4, h=2.8)
nt.bar((100 * df.persons.clip(upper=6).value_counts(normalize=True)).sort_index(), "persons (6 = 6+)", "share (%)", ax=ax[0])
nt.hist(df.income_2016usd.clip(upper=300000) / 1000, "household income (2016 k$)", "households", bins=50, ax=ax[1])
nt.bar((100 * df.children_any.value_counts(normalize=True)).sort_index(), "children (1 yes)", "share (%)", ax=ax[2])
nt.hist(df.age_oldest, "age of the oldest member (years)", "households", bins=range(15, 96, 5), ax=ax[3])
plt.tight_layout(); plt.show()
pz = pd.read_csv(os.path.join(SH, "resloc_work", "geo_puma_zone_all.csv")); dest = df.puma7.value_counts()
z = pz.assign(n=pz.puma7.map(dest).fillna(0) * pz.hu / pz.groupby("puma7").hu.transform("sum")).groupby("zone").n.sum()
fig, ax = plt.subplots(figsize=(7.5, 6)); nt.zone_map(z[z > 0], "movers (PUMA of the dwelling spread over its zones)", ax=ax, log=True, zoom="md"); plt.tight_layout(); plt.show()
t = pd.concat({"structure type": nt.half_comparison(df.type_0, HO.type_0, "x"),
               "bedrooms": nt.half_comparison(df.bedrooms_0.clip(upper=5), HO.bedrooms_0.clip(upper=5), "x")})
t

Mover households (persons, income, children, age), their PUMAs, and the shares of the two halves (estimation 80 %, hold-out 20 %).

In [ ]:
# Descriptive statistics
alts = nt.alt_columns(df)          # alternative-specific columns <variable>_<j>, used in section 6

### 4.1 Variables in the data

Each explanatory variable of the final specification as it enters the utility, interactions included (read from the model's own utility functions: the part of the utility that each coefficient multiplies; coefficients that share a label, e.g. the copies of a term for several purposes, are shown in one panel). Here: the chosen dwelling against the 29 sampled dwellings of the same PUMA; unweighted.

In [ ]:
# every explanatory variable of the specification: chosen against sampled alternatives
import varplots as vp
_db, _ll = M.model(); U_, AV_, CH_ = vp.find_choice(_ll)
VP = vp.location_from_util(U_, AV_, CH_, df, vp.silo_labels(MODEL_FOLDER), sampled=True)
print(f"{len(VP)} variables, {len(df):,} movers")
vp.location_figures(VP, what='sampled', unit='movers')

How to read it: dark grey line = the chosen alternatives, light bars = the sampled alternatives, as a share of each (%); dashed lines are the two means; 0/1 variables are shown as the share with value 1. A continuous variable that is zero at every alternative of a record cannot affect that record's choice, so such records are left out of its panel (e.g. an interaction with a person group; the share kept is in the panel title). The further apart the two, the more the variable separates the choices.

### 4.2 New dwellings, bedrooms and housing cost

The chosen dwelling against the 29 sampled dwellings: built in the last five years, bedrooms against the bedrooms needed (household size, at most 5, as in the model: shortfall, exact, extra), and the housing cost as a share of income by income group (2016 $).

In [ ]:
# new dwellings, bedrooms against need, cost share of income; chosen against sampled
import varplots as vp
vp.dwelling_figure(df)

## 5. Model specification

*The utility functions, read from the model script; nothing is re-specified here.*

The specification is the folder's script `model/estimate_residential_dwelling.py`, imported as it is. The start values are the article estimates.

In [ ]:
# Model specification
print(inspect.getsource(M.model))

In [ ]:
# Model specification
t0 = time.time()
database, ll = M.model()
used = nt.model_variables(ll)
print(f"database: {database.num_rows():,} rows; {len(used)} data columns used by the model; built in {time.time()-t0:.1f} s")

## 6. Correlation of the explanatory variables

*Correlation of the explanatory variables. Look for: |r| of 0.7 or more and VIF of 10 or more; these make single coefficients less precise.*

Correlations of the variables that enter the utility, computed over the alternatives available to each decision maker
(alternative-specific variables) and over decision makers (person and household variables). |r| ≥ 0.7 and VIF ≥ 10 are
listed. Interaction terms are correlated with their main effects by construction.

In [ ]:
# Correlation of the explanatory variables
stems = sorted({c.rsplit("_", 1)[0] for c in used if c.rsplit("_", 1)[0] in alts and c.rsplit("_", 1)[-1].isdigit()}
               - {"av", "chosen_set", "zone", "stype"})
long = nt.explanatory(nt.stack_alternatives(df, stems))
print(f"{len(stems)} alternative-specific variables, {len(long):,} available alternatives")
Ca = nt.corr_heatmap(long, "Alternative-specific variables (available alternatives)")
display(nt.high_pairs(Ca)); display(nt.vif(long))
plain = [c for c in used if c in df.columns and c not in sum(alts.values(), []) and c not in ("choice", "weight")]
if len(plain) > 1:
    Xp = nt.explanatory(df[plain]); Cp = nt.corr_heatmap(Xp, "Decision-maker variables"); display(nt.high_pairs(Cp))

## 7. Log-likelihood at the article coefficients

*Log-likelihood at the article coefficients, without estimation. Look for: the two numbers should be equal (to rounding).*

No estimation: the log-likelihood at the start values (the article estimates). Target: article -47478.6,
Biogeme -47478.5882 (N = 15163, K = 18).

In [ ]:
# Log-likelihood at the article coefficients
if RUN_CHECK:
    p = Parameters(); p.set_value("generate_html", False); p.set_value("generate_yaml", False)
    here = os.getcwd(); os.chdir(__import__("tempfile").mkdtemp())
    t0 = time.time()
    b = bio.BIOGEME(database, ll, parameters=p); b.model_name = NAME + "_check"
    ll0 = b.calculate_init_likelihood()
    os.chdir(here)
    print(f"N {database.num_rows():,}   LL at the article coefficients: {ll0:,.4f}   (target -47478.5882)   {time.time()-t0:.0f} s")
else:
    print("skipped (RUN_CHECK = False)")

## 8. Biogeme estimation

*Biogeme estimation, only with RUN_ESTIMATION. Look for: convergence and the run time.*

The same call as `model/estimate_residential_dwelling.py`: Biogeme with its defaults (read from `output/biogeme.toml`), start values = article estimates.
Report written to `output/residential_dwelling.html`, `.yaml`, and `output/residential_dwelling_estimates.csv`.

In [ ]:
# Biogeme estimation
if RUN_ESTIMATION:
    here = os.getcwd(); os.chdir(OUT)
    t0 = time.time()
    b = bio.BIOGEME(database, ll); b.model_name = NAME
    results = b.estimate()
    os.chdir(here)
    print(f"estimation time {time.time()-t0:.0f} s"); print(results.short_summary())
    nt.estimates_table(results)[["parameter", "coefficient", "robust_se", "t", "p"]].rename(
        columns={"coefficient": "Value", "robust_se": "Robust std err.", "t": "Robust t-stat.", "p": "Robust p-value", "parameter": "Name"}
    ).to_csv(os.path.join(OUT, f"{NAME}_estimates.csv"), index=False)
else:
    ref_time = os.path.getmtime(os.path.join(FOLDER, "model", "reference_estimates.csv"))
    y = [f for f in sorted(glob.glob(os.path.join(OUT, f"{NAME}*.yaml")), key=os.path.getmtime) if os.path.getmtime(f) > ref_time and "_check" not in f]
    results = EstimationResults.from_yaml_file(y[-1]) if y else None
    print(f"RUN_ESTIMATION = False: read {os.path.basename(y[-1])} (newer than model/reference_estimates.csv)" if y else "RUN_ESTIMATION = False and no current result in output/: sections 9-10 are skipped")

## 9. Results

*The estimates and what they mean. Look for: coefficients equal to the article within rounding, and the readings below.*

### Estimates in the article format
Coefficient (stars: *** p < 0.01, ** p < 0.05, * p < 0.10, as in the article), robust s.e., p.

In [ ]:
# Results
if results is not None:
    est_t = nt.estimates_table(results)
    display(nt.article_format(est_t))

### What the coefficients say

One plain line per term, from your estimates (from the article estimates if there are none). Income interactions are given at $25k, $50k and $100k; odds change = exp(coefficient) per unit.

In [ ]:
# plain readings of the coefficients and the coefficient plot (95 % robust intervals, grouped by block)
est_v = est_t if results is not None else rd.reference(MODEL_FOLDER)[0]
display(rd.readings(MODEL_FOLDER, est_v).style.hide(axis="index").set_properties(**{"text-align": "left"}))
fg.coef_figure(fg.coef_rows(MODEL_FOLDER, est_v)); plt.show()

How to read it: a point right of zero makes the alternative more likely; an interval that crosses zero is not significant.

### Effects by income

Main term plus its income interaction, with 95 % delta-method bands from the robust covariance.

In [ ]:
# effect on utility by household income (or income group), 95 % delta-method bands
b = results.get_beta_values() if results is not None else dict(zip(est_v.parameter, est_v.coefficient))
names_ = list(b); V = pd.DataFrame(np.asarray(results.robust_variance_covariance_matrix), index=names_, columns=names_) if results is not None else None
fg.income_effect_figure(MODEL_FOLDER, b, V); plt.show()

How to read it: above zero the feature makes a dwelling more attractive (structure types: than a detached house), below zero less; where the band crosses zero the effect is not significant at that income.

### Goodness of fit

In [ ]:
# Goodness of fit
if results is not None:
    display(nt.fit_statistics(results))

### Comparison with the article (`model/reference_estimates.csv`)

In [ ]:
# Comparison with the article (model/reference_estimates.csv)
if results is not None:
    REF = os.path.join(FOLDER, "model", "reference_estimates.csv")
    cmp = nt.compare(est_t, REF)
    cmp.to_csv(os.path.join(OUT, "comparison_with_article_notebook.csv"), index=False)
    print(f"{int(cmp.matches_article.sum())} of {len(cmp)} parameters match the article (coefficient and robust s.e. within the 4th-decimal rounding); "
          f"largest |difference|: coefficient {cmp.diff_coef.abs().max():.5f}, s.e. {cmp.diff_se.abs().max():.5f}")
    display(cmp.round({"coefficient": 4, "robust_se": 4, "p": 4, "diff_coef": 5, "diff_se": 5}))
    stats = nt.article_statistics(REF)
    if len(stats):
        stats["Biogeme"] = stats["statistic"].map({"LL(beta)": results.final_loglikelihood, "N": database.num_rows() if "database" in dir() else None,
                                                   "K": results.number_of_free_parameters})
        display(stats)
    nt.plot_against_article(cmp, RUN.get("model", NAME))

### Correlation of the estimates
Robust correlation matrix of the estimated parameters; pairs with |r| ≥ 0.7 listed.

In [ ]:
# Correlation of the estimates
if results is not None:
    display(nt.parameter_correlation(results, "Robust correlation of the estimates"))

## 10. Hold-out sample

*Fit on the hold-out records, which the estimation did not use. Look for: a log-likelihood per record close to that of the estimation sample, and the maps.*

In [ ]:
# Hold-out sample
if RUN_HOLDOUT:
    import tempfile
    est_data = M.DATA
    M.DATA = os.path.join(FOLDER, "input", "data", "holdout", 'dwelling_holdout.parquet')
    at = "your estimates" if results is not None else "the article coefficients"
    try:
        hdb, hll = M.model(start=results.get_beta_values() if results is not None else None)
    finally:
        M.DATA = est_data
    p = Parameters(); p.set_value("generate_html", False); p.set_value("generate_yaml", False)
    here = os.getcwd(); os.chdir(tempfile.mkdtemp())
    b = bio.BIOGEME(hdb, hll, parameters=p); b.model_name = NAME + "_holdout"
    llh = b.calculate_init_likelihood(); os.chdir(here)
    print(f"hold-out: N {hdb.num_rows():,}   LL at {at} {llh:,.2f}   mean LL per observation {llh/hdb.num_rows():.4f}")
    if results is not None:
        print(f"estimation sample: mean LL per observation {results.final_loglikelihood/database.num_rows():.4f}")
else:
    print("skipped (RUN_HOLDOUT = False)")

### Hold-out shares by structure type

Share of the hold-out movers' chosen dwellings by structure type, observed and predicted (sum over the 30 alternatives of probability x type).

In [ ]:
# predicted against observed structure type of the chosen dwelling, hold-out movers (article coefficients if there are no estimates)
est_data = M.DATA; M.DATA = os.path.join(FOLDER, "input", "data", "holdout", "dwelling_holdout.parquet")
try:
    hdb, hll = M.model(); hdf = pd.read_parquet(M.DATA)
finally:
    M.DATA = est_data
ref_ = rd.reference(MODEL_FOLDER)[0]
betas = results.get_beta_values() if results is not None else dict(zip(ref_.parameter, ref_.coefficient))
P = nt.predicted_probabilities(hdb, hll, betas)
fg.obs_model_figure(fg.dwelling_holdout_panels(hdf, P)); plt.show()

How to read it: dark grey observed, orange model: (a) structure type of the chosen dwelling, (b) share in multifamily and (c) mean monthly cost by income group; equal bars mean the model reproduces the hold-out.

## Glossary

- **PUMA**: Public Use Microdata Area: a Census area of about 100,000 people, the smallest area of the ACS microdata.
- **sampled alternatives**: The chosen alternative and others drawn at random with equal probability; this gives the same estimates as the full set, with less precision.
- **inclusive value**: The expected value of the best dwelling in a PUMA: ln of the mean of exp(V) over 200 sampled dwellings. Its coefficient is held at 1.
- **log-likelihood**: Sum over decision makers of ln(probability of the observed choice). Closer to 0 is better.
- **BIC**: Bayesian information criterion: -2 log-likelihood + K ln N. Lower is better; it penalises extra parameters.
- **hold-out**: The 20 % (residential, cars) or half (survey models) of the records not used for estimation. It was also used in the specification search, so its fit is not a fully independent test.
- **robust s.e.**: Standard error from the sandwich formula, valid when the model is not exactly right.
- **VIF**: Variance inflation factor: how much the correlation with other variables inflates a coefficient's variance; 10 or more is high.